# Lab 4: Working with HuggingFace LLMs

## Introduction

In this lab, we'll explore how to work with Large Language Models (LLMs) from HuggingFace. We'll focus on smaller models (up to 7B parameters) that are more accessible for learning and experimentation while still providing impressive capabilities. We'll also dive into key parameters that control text generation and explore various interesting use cases.

## 1. Setup and Environment Preparation

Let's start by installing the necessary libraries:

In [ ]:
# Install required packages
!pip install transformers accelerate bitsandbytes datasets evaluate scipy scikit-learn sentencepiece
!pip install torch torchvision

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from transformers import AutoModelForSeq2SeqLM, AutoModelForSequenceClassification
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset
import time

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# If using GPU, print some information about it
if device == "cuda":
    print(f"Device name: {torch.cuda.get_device_name(0)}")
    print(f"Memory allocated: {torch.cuda.memory_allocated(0) / 1024 ** 3:.2f} GB")
    print(f"Memory reserved: {torch.cuda.memory_reserved(0) / 1024 ** 3:.2f} GB")

## 2. Understanding LLM Parameters

Before we start working with models, let's understand the key parameters that control text generation:

### a) Temperature

Temperature controls the randomness or creativity of the model's outputs. 

- **Lower temperature** (e.g., 0.1): More focused, deterministic, and conservative outputs
- **Higher temperature** (e.g., 1.5): More diverse, creative, and sometimes unexpected outputs
- **Default** (1.0): Balanced outputs

Let's visualize how temperature affects token probability distribution:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Example logits (raw model outputs before softmax)
logits = np.array([2.5, 1.0, 0.2, -0.5, -1.0])
token_names = ["the", "a", "an", "in", "on"]

temperatures = [0.2, 0.5, 1.0, 2.0]
plt.figure(figsize=(12, 8))

for i, temp in enumerate(temperatures):
    # Apply temperature scaling and softmax
    scaled_logits = logits / temp
    probs = np.exp(scaled_logits) / np.sum(np.exp(scaled_logits))
    
    plt.subplot(2, 2, i+1)
    plt.bar(token_names, probs)
    plt.title(f"Temperature = {temp}")
    plt.ylabel("Probability")
    plt.ylim(0, 1)

plt.tight_layout()
plt.show()

### b) Max Tokens (max_length)

This parameter sets the maximum length of the generated text:

- Too small: Incomplete responses
- Too large: Unnecessarily long responses and increased computation time
- Optimal: Depends on your use case, but typically ranges from 50-1000 tokens

### c) Top-k Sampling

Top-k sampling restricts the model to consider only the top-k most likely tokens at each step:

- Small k (e.g., 5-10): More focused, less diverse outputs
- Large k (e.g., 50): More diverse but potentially less coherent outputs
- Special case: k=1 is greedy decoding (always choose the most likely token)

### d) Top-p (nucleus) Sampling

Top-p sampling dynamically selects the smallest set of tokens whose cumulative probability exceeds p:

- Small p (e.g., 0.5): More deterministic outputs
- Large p (e.g., 0.95): More diverse outputs
- Typically used together with top-k

Let's visualize top-k and top-p sampling:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Example token probabilities after softmax
probs = np.array([0.35, 0.2, 0.15, 0.1, 0.05, 0.05, 0.03, 0.03, 0.02, 0.02])
token_names = ["the", "a", "to", "in", "is", "and", "of", "that", "it", "with"]

# Sort by probability for visualization
sorted_indices = np.argsort(probs)[::-1]
sorted_probs = probs[sorted_indices]
sorted_tokens = [token_names[i] for i in sorted_indices]

# Calculate cumulative probabilities
cumulative_probs = np.cumsum(sorted_probs)

# Visualize Top-k and Top-p
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Top-k (k=5)
k = 5
colors = ['#1f77b4' if i < k else '#d3d3d3' for i in range(len(sorted_probs))]
ax1.bar(sorted_tokens, sorted_probs, color=colors)
ax1.set_title(f"Top-k Sampling (k={k})")
ax1.set_ylabel("Probability")
ax1.tick_params(axis='x', rotation=45)

# Top-p (p=0.8)
p = 0.8
top_p_indices = cumulative_probs <= p
included_tokens = np.sum(top_p_indices) + 1  # +1 because we need to include the first token that exceeds p
colors = ['#1f77b4' if i < included_tokens else '#d3d3d3' for i in range(len(sorted_probs))]
ax2.bar(sorted_tokens, sorted_probs, color=colors)
ax2.plot(sorted_tokens, cumulative_probs, 'ro-', linewidth=2)
ax2.axhline(y=p, color='r', linestyle='--', label=f"p={p}")
ax2.set_title(f"Top-p Sampling (p={p})")
ax2.set_ylabel("Probability")
ax2.tick_params(axis='x', rotation=45)
ax2.legend()

plt.tight_layout()
plt.show()

## 3. Loading Smaller LLMs from HuggingFace

Now, let's load some smaller but powerful models from HuggingFace. These models are up to 7B parameters, making them more accessible for experimentation.

### a) TinyLlama (1.1B parameters)

TinyLlama is a compact yet capable model that's perfect for learning:

In [ ]:
# Load TinyLlama
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Load with 4-bit quantization to reduce memory usage
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    load_in_4bit=True
)

print(f"Model loaded: {model_name}")
print(f"Model size: ~1.1B parameters")

Let's create a helper function to generate text with different parameters:

In [ ]:
def generate_text(prompt, temperature=0.7, max_length=100, top_k=50, top_p=0.95, model=model, tokenizer=tokenizer):
    """
    Generate text using the loaded model with specified parameters.
    
    Args:
        prompt (str): The input prompt
        temperature (float): Controls randomness (higher = more random)
        max_length (int): Maximum number of tokens to generate
        top_k (int): Number of highest probability tokens to consider
        top_p (float): Cumulative probability threshold for nucleus sampling
        model: The language model
        tokenizer: The tokenizer
        
    Returns:
        str: The generated text
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    # Record start time
    start_time = time.time()
    
    # Generate text
    output = model.generate(
        inputs["input_ids"],
        max_length=max_length,
        temperature=temperature,
        top_k=top_k,
        top_p=top_p,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    
    # Record end time
    end_time = time.time()
    generation_time = end_time - start_time
    
    # Decode the output
    generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
    
    print(f"Generation took {generation_time:.2f} seconds")
    return generated_text

In [ ]:
prompt = "Explain the concept of transformer models in simple terms:"
generated_text = generate_text(prompt)
print(generated_text)

### b) Phi-2 (2.7B parameters)

Microsoft's Phi-2 is an excellent small model with strong performance:

In [ ]:
# Load Phi-2
model_name = "microsoft/phi-2"

phi_tokenizer = AutoTokenizer.from_pretrained(model_name)
phi_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    load_in_4bit=True
)

print(f"Model loaded: {model_name}")
print(f"Model size: ~2.7B parameters")

### c) Mistral-7B (7B parameters)

Mistral is a state-of-the-art 7B parameter model with impressive capabilities:

In [ ]:
# Load Mistral-7B
model_name = "mistralai/Mistral-7B-Instruct-v0.2"

mistral_tokenizer = AutoTokenizer.from_pretrained(model_name)
mistral_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    load_in_4bit=True
)

print(f"Model loaded: {model_name}")
print(f"Model size: ~7B parameters")

## 4. Exploring Parameter Effects on Text Generation

Now, let's see how different parameters affect text generation using our models:

In [ ]:
prompt = "Write a short poem about artificial intelligence:"

print("🔍 Low Temperature (0.3) - More Focused:")
generate_text(prompt, temperature=0.3, model=mistral_model, tokenizer=mistral_tokenizer)

print("\n🔍 High Temperature (1.5) - More Creative:")
generate_text(prompt, temperature=1.5, model=mistral_model, tokenizer=mistral_tokenizer)

print("\n🔍 Low Top-k (5) - Limited Options:")
generate_text(prompt, top_k=5, model=mistral_model, tokenizer=mistral_tokenizer)

print("\n🔍 Low Top-p (0.5) - More Deterministic:")
generate_text(prompt, top_p=0.5, model=mistral_model, tokenizer=mistral_tokenizer)

Let's create a function to compare generation times across different models:

In [ ]:
def compare_model_speeds(prompt, models_dict, max_length=100):
    """
    Compare generation speeds across different models.
    
    Args:
        prompt (str): Input prompt
        models_dict (dict): Dictionary mapping model names to (model, tokenizer) tuples
        max_length (int): Maximum generation length
        
    Returns:
        pd.DataFrame: Dataframe with generation times
    """
    results = []
    
    for model_name, (model, tokenizer) in models_dict.items():
        # Tokenize input
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        
        # Record start time
        start_time = time.time()
        
        # Generate text
        output = model.generate(
            inputs["input_ids"],
            max_length=max_length,
            do_sample=True,
            temperature=0.7,
            pad_token_id=tokenizer.eos_token_id
        )
        
        # Record end time
        end_time = time.time()
        generation_time = end_time - start_time
        
        # Save result
        results.append({
            "Model": model_name,
            "Generation Time (seconds)": generation_time,
            "Output Length (tokens)": output.shape[1]
        })
    
    # Create DataFrame
    return pd.DataFrame(results)

# Compare models
models_to_compare = {
    "TinyLlama (1.1B)": (model, tokenizer),
    "Phi-2 (2.7B)": (phi_model, phi_tokenizer),
    "Mistral (7B)": (mistral_model, mistral_tokenizer)
}

comparison = compare_model_speeds(
    "Explain the benefits and limitations of large language models:", 
    models_to_compare
)
print(comparison)

In [ ]:
# Visualize comparison
plt.figure(figsize=(10, 6))
plt.barh(comparison["Model"], comparison["Generation Time (seconds)"])
plt.xlabel("Generation Time (seconds)")
plt.title("Model Generation Speed Comparison")
plt.tight_layout()
plt.show()

## 5. Interesting Use Cases with HuggingFace LLMs

Now let's explore some interesting use cases for these smaller but powerful LLMs:

### a) Chain-of-Thought Reasoning

Let's see how the model performs with chain-of-thought prompting:

In [ ]:
cot_prompt = """
Q: Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many tennis balls does he have now?

A: Let's think through this step by step:
1. Roger starts with 5 tennis balls.
2. He buys 2 cans of tennis balls.
3. Each can has 3 tennis balls.
4. So the 2 cans contain 2 × 3 = 6 tennis balls.
5. In total, Roger has 5 + 6 = 11 tennis balls.

Q: Sarah is building a brick wall. She has already placed 23 bricks. Each box contains 48 bricks. She has 3 full boxes and one box with 12 bricks. How many bricks does she have left to place?

A:
"""

print("Chain-of-Thought Reasoning Example:")
generate_text(cot_prompt, max_length=300, model=mistral_model, tokenizer=mistral_tokenizer)

### b) Text Classification with Smaller Models

Let's use a smaller model for sentiment analysis:

In [ ]:
# Load a sentiment analysis model
sentiment_model_name = "distilbert-base-uncased-finetuned-sst-2-english"
sentiment_classifier = pipeline(
    "sentiment-analysis", 
    model=sentiment_model_name, 
    device=0 if device == "cuda" else -1
)

# Test on some examples
texts = [
    "I absolutely loved the movie, it was fantastic!",
    "The service was terrible and the food was cold.",
    "The product works as expected, nothing special.",
    "I can't believe how amazing this new phone is, it exceeded all my expectations!"
]

results = sentiment_classifier(texts)
for text, result in zip(texts, results):
    print(f"Text: {text}")
    print(f"Sentiment: {result['label']}, Score: {result['score']:.4f}")
    print()

In [ ]:
# Visualize confidence scores
labels = [result['label'] for result in results]
scores = [result['score'] for result in results]
colors = ['green' if label == 'POSITIVE' else 'red' for label in labels]

plt.figure(figsize=(12, 6))
plt.barh(range(len(texts)), scores, color=colors)
plt.yticks(range(len(texts)), [text[:50] + '...' if len(text) > 50 else text for text in texts])
plt.xlabel('Confidence Score')
plt.title('Sentiment Analysis Results')
plt.tight_layout()
plt.show()

### c) Text Summarization

Let's try text summarization with a smaller model:

In [ ]:
# Load a summarization model
summarizer = pipeline(
    "summarization", 
    model="sshleifer/distilbart-cnn-6-6", 
    device=0 if device == "cuda" else -1
)

# Example long text
long_text = """
Artificial intelligence (AI) is intelligence demonstrated by machines, as opposed to intelligence displayed by animals and humans. AI research has been defined as the field of study of intelligent agents, which refers to any system that perceives its environment and takes actions that maximize its chance of achieving its goals.

The term "artificial intelligence" had previously been used to describe machines that mimic and display "human" cognitive skills that are associated with the human mind, such as "learning" and "problem-solving". This definition has since been rejected by major AI researchers who now describe AI in terms of rationality and acting rationally, which does not limit how intelligence can be articulated.

AI applications include advanced web search engines (e.g., Google), recommendation systems (used by YouTube, Amazon, and Netflix), understanding human speech (such as Siri and Alexa), self-driving cars (e.g., Waymo), generative or creative tools (ChatGPT and AI art), automated decision-making, and competing at the highest level in strategic game systems (such as chess and Go).

As machines become increasingly capable, tasks considered to require "intelligence" are often removed from the definition of AI, a phenomenon known as the AI effect. For instance, optical character recognition is frequently excluded from things considered to be AI, having become a routine technology.
"""

summary = summarizer(long_text, max_length=150, min_length=50, do_sample=False)
print("Original Text Length:", len(long_text.split()))
print("Summary Length:", len(summary[0]['summary_text'].split()))
print("\nSummary:")
print(summary[0]['summary_text'])

### d) Zero-shot and Few-shot Learning

Let's explore how smaller LLMs handle zero-shot and few-shot tasks:

In [ ]:
# Zero-shot classification
def zero_shot_classification(model, tokenizer, text, categories):
    # Format prompt for zero-shot classification
    prompt = f"""Classify the following text into one of these categories: {', '.join(categories)}.
    
Text: \"{text}\"
Category:"""
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(
        inputs["input_ids"],
        max_length=len(inputs["input_ids"][0]) + 10,
        temperature=0.3,
        do_sample=False
    )
    result = tokenizer.decode(output[0], skip_special_tokens=True)
    
    # Extract the predicted category
    predicted = result.split("Category:")[-1].strip()
    
    return predicted

In [ ]:
# Few-shot learning example
few_shot_prompt = """Here are some examples of text and their sentiment:

Text: "I love this product, it works great!"
Sentiment: Positive

Text: "This was a terrible experience, would not recommend."
Sentiment: Negative

Text: "The service was okay, not great but not bad either."
Sentiment: Neutral

Text: "I can't believe how poorly this was handled."
Sentiment: """

# Test texts for zero-shot
test_texts = [
    "The new climate policy is ambitious but lacks concrete implementation details.",
    "The latest smartphone features breakthrough technology that will revolutionize the industry."
]

categories = ["Politics", "Technology", "Entertainment", "Sports", "Health"]

print("Zero-shot Classification Examples:")
for text in test_texts:
    category = zero_shot_classification(mistral_model, mistral_tokenizer, text, categories)
    print(f"Text: {text}")
    print(f"Predicted Category: {category}")
    print()

In [ ]:
# Test few-shot learning
print("Few-shot Learning Example:")
few_shot_result = generate_text(few_shot_prompt + "I waited 2 hours for my food and it was cold when it arrived.", 
                              temperature=0.3, 
                              model=mistral_model, 
                              tokenizer=mistral_tokenizer)
print(few_shot_result)

### e) Creative Writing Assistant

Let's create a simple creative writing assistant:

In [ ]:
def writing_assistant(scenario, style, length="short", model=mistral_model, tokenizer=mistral_tokenizer):
    """
    Generate creative writing based on a scenario and style.
    
    Args:
        scenario (str): Brief description of the scenario
        style (str): Writing style (e.g., "humorous", "noir", "fantasy")
        length (str): Desired length ("short", "medium", "long")
        model: The language model
        tokenizer: The tokenizer
        
    Returns:
        str: Generated creative text
    """
    # Map length to token count
    length_map = {
        "short": 150,
        "medium": 300,
        "long": 500
    }
    max_length = length_map.get(length, 200)
    
    prompt = f"""Write a {style} {length} story about the following scenario:
    
Scenario: {scenario}

Story:"""
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(
        inputs["input_ids"],
        max_length=len(inputs["input_ids"][0]) + max_length,
        temperature=0.8,
        top_p=0.9,
        do_sample=True
    )
    story = tokenizer.decode(output[0], skip_special_tokens=True)
    
    # Extract the story part
    return story.split("Story:")[-1].strip()

In [ ]:
# Test the writing assistant
scenario = "A detective discovers that the city's mayor is hiding a strange secret"
style = "noir"
length = "short"

print(f"Writing a {style} {length} story about: {scenario}")
story = writing_assistant(scenario, style, length)
print(story)

## 6. Model Evaluation and Comparison

Let's evaluate our models on some common benchmarks:

In [ ]:
def evaluate_accuracy(model, tokenizer, questions, reference_answers):
    """
    Simple accuracy evaluation for question answering.
    
    Args:
        model: The language model
        tokenizer: The tokenizer
        questions (list): List of questions
        reference_answers (list): List of reference answers
        
    Returns:
        dict: Evaluation metrics
    """
    correct = 0
    response_times = []
    
    for q, ref_answer in zip(questions, reference_answers):
        prompt = f"Q: {q}\nA:"
        
        # Time the response
        start_time = time.time()
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        output = model.generate(
            inputs["input_ids"],
            max_length=len(inputs["input_ids"][0]) + 50,
            temperature=0.3,
            do_sample=False
        )
        end_time = time.time()
        
        # Process output
        response_time = end_time - start_time
        response_times.append(response_time)
        
        prediction = tokenizer.decode(output[0], skip_special_tokens=True)
        prediction = prediction.split("A:")[-1].strip().lower()
        
        # Very simple exact match checking - could be improved with more sophisticated metrics
        if any(ans.lower() in prediction for ans in ref_answer.split('|')):
            correct += 1
    
    return {
        "accuracy": correct / len(questions),
        "avg_response_time": sum(response_times) / len(response_times)
    }

In [ ]:
# Simple evaluation dataset
eval_questions = [
    "What is the capital of France?",
    "Who wrote 'Romeo and Juliet'?",
    "What is 7 x 8?",
    "What is the chemical symbol for gold?",
    "What year did World War II end?"
]

eval_answers = [
    "Paris",
    "William Shakespeare|Shakespeare",
    "56",
    "Au",
    "1945"
]

# Evaluate each model
models_to_evaluate = {
    "TinyLlama (1.1B)": (model, tokenizer),
    "Phi-2 (2.7B)": (phi_model, phi_tokenizer),
    "Mistral (7B)": (mistral_model, mistral_tokenizer)
}

evaluation_results = {}
for model_name, (model, tokenizer) in models_to_evaluate.items():
    print(f"Evaluating {model_name}...")
    results = evaluate_accuracy(model, tokenizer, eval_questions, eval_answers)
    evaluation_results[model_name] = results
    print(f"  Accuracy: {results['accuracy']:.2f}")
    print(f"  Avg Response Time: {results['avg_response_time']:.2f} seconds")

In [ ]:
# Visualize evaluation results
accuracies = [results['accuracy'] for results in evaluation_results.values()]
response_times = [results['avg_response_time'] for results in evaluation_results.values()]
model_names = list(evaluation_results.keys())

fig, ax1 = plt.subplots(figsize=(10, 6))

color = 'tab:blue'
ax1.set_xlabel('Model')
ax1.set_ylabel('Accuracy', color=color)
ax1.bar([i-0.2 for i in range(len(model_names))], accuracies, 0.4, color=color, alpha=0.7)
ax1.tick_params(axis='y', labelcolor=color)
ax1.set_ylim(0, 1.1)

ax2 = ax1.twinx()
color = 'tab:red'
ax2.set_ylabel('Response Time (seconds)', color=color)
ax2.bar([i+0.2 for i in range(len(model_names))], response_times, 0.4, color=color, alpha=0.7)
ax2.tick_params(axis='y', labelcolor=color)

plt.xticks(range(len(model_names)), model_names, rotation=45)
plt.title('Model Comparison: Accuracy vs Response Time')
plt.tight_layout()
plt.show()

## 7. Advanced Topic: Model Quantization and Optimization

Quantization is crucial for running these models efficiently:

In [ ]:
def compare_quantization_options():
    """
    Compare different quantization options and their impact on memory and speed.
    This is a demonstration function - we don't actually load all variants.
    """
    quantization_options = [
        {"name": "FP32 (No Quantization)", "bits": 32, "memory": "~28 GB", "relative_speed": 1.0, "quality": "Baseline"},
        {"name": "FP16", "bits": 16, "memory": "~14 GB", "relative_speed": 1.5, "quality": "Minimal Loss"},
        {"name": "BF16", "bits": 16, "memory": "~14 GB", "relative_speed": 1.7, "quality": "Minimal Loss"},
        {"name": "INT8", "bits": 8, "memory": "~7 GB", "relative_speed": 2.0, "quality": "Slight Loss"},
        {"name": "4-bit (GPTQ)", "bits": 4, "memory": "~3.5 GB", "relative_speed": 1.8, "quality": "Moderate Loss"},
        {"name": "4-bit (AWQ)", "bits": 4, "memory": "~3.5 GB", "relative_speed": 2.2, "quality": "Minor Loss"},
        {"name": "3-bit", "bits": 3, "memory": "~2.6 GB", "relative_speed": 2.5, "quality": "Significant Loss"},
        {"name": "2-bit", "bits": 2, "memory": "~1.8 GB", "relative_speed": 3.0, "quality": "Major Loss"}
    ]
    
    return pd.DataFrame(quantization_options)

quant_comparison = compare_quantization_options()
print(quant_comparison)

In [ ]:
# Visualization of memory requirements
plt.figure(figsize=(12, 6))
plt.bar(quant_comparison["name"], [float(m.split()[0].replace("~", "")) for m in quant_comparison["memory"]])
plt.ylabel("Memory Usage (GB)")
plt.title("Memory Requirements for Different Quantization Methods (7B Model)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 8. Practical Applications

Let's explore some practical applications of these smaller LLMs:

### a) Document Q&A System

In [ ]:
def document_qa(document_text, question, model=mistral_model, tokenizer=mistral_tokenizer):
    """
    Simple document Q&A using a language model.
    
    Args:
        document_text (str): The document text
        question (str): The question to answer
        model: The language model
        tokenizer: The tokenizer
        
    Returns:
        str: The answer to the question
    """
    # Truncate document if too long
    if len(document_text.split()) > 500:
        document_text = " ".join(document_text.split()[:500]) + "..."
    
    prompt = f"""Document: {document_text}

Question: {question}

Answer:"""
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(
        inputs["input_ids"],
        max_length=len(inputs["input_ids"][0]) + 100,
        temperature=0.3,
        do_sample=False
    )
    
    response = tokenizer.decode(output[0], skip_special_tokens=True)
    answer = response.split("Answer:")[-1].strip()
    
    return answer

In [ ]:
# Example document
sample_document = """
Artificial General Intelligence (AGI) refers to highly autonomous systems that outperform humans at most economically valuable work. Developing AGI would be a profound change in human history, with possible risks and benefits that are extraordinarily difficult to anticipate and prepare for.

There are three major technical approaches to AGI development:
1. Symbol manipulation: Systems that process abstract symbols according to explicitly programmed rules.
2. Connectionism: Artificial neural networks that learn by adjusting connection weights.
3. Hybrid approaches: Combinations of symbolic and connectionist methods.

Recent progress in AI has been driven by deep learning, a connectionist approach. Key advancements include transformer models like GPT and multimodal systems capable of processing text, images, and other data types.

Safety research is a crucial component of AGI development. This includes work on interpretability, robustness, alignment, and governance. Without adequate safety measures, advanced AI systems could cause harm if they pursue goals misaligned with human values.
"""

# Test questions
qa_questions = [
    "What are the three major technical approaches to AGI development?",
    "What has driven recent progress in AI?",
    "Why is safety research important for AGI?"
]

print("Document Q&A Example:")
for question in qa_questions:
    answer = document_qa(sample_document, question)
    print(f"Q: {question}")
    print(f"A: {answer}\n")

### b) Code Assistant

In [ ]:
def code_assistant(task_description, programming_language="python", model=mistral_model, tokenizer=mistral_tokenizer):
    """
    Generate code based on a task description.
    
    Args:
        task_description (str): Description of the coding task
        programming_language (str): The target programming language
        model: The language model
        tokenizer: The tokenizer
        
    Returns:
        str: Generated code
    """
    prompt = f"""Write {programming_language} code for the following task:

Task: {task_description}

Here is the {programming_language} code:
```{programming_language}
"""
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(
        inputs["input_ids"],
        max_length=len(inputs["input_ids"][0]) + 300,
        temperature=0.2,
        do_sample=True,
        top_p=0.9
    )
    
    response = tokenizer.decode(output[0], skip_special_tokens=True)
    
    # Extract code between backticks
    code_pattern = f"```{programming_language}\n(.*?)```"
    import re
    match = re.search(code_pattern, response, re.DOTALL)
    
    if match:
        return match.group(1).strip()
    else:
        # If pattern not found, return everything after "Here is the code:"
        return response.split(f"Here is the {programming_language} code:")[-1].strip()

In [ ]:
# Test the code assistant
code_tasks = [
    "Create a function to check if a string is a palindrome",
    "Write a simple Flask API with one endpoint that returns the current time"
]

print("Code Assistant Examples:")
for task in code_tasks:
    code = code_assistant(task)
    print(f"Task: {task}")
    print("Generated Code:")
    print(f"```python\n{code}\n```\n")

## 9. Ethical Considerations and Responsible Use

It's important to discuss the ethical considerations when working with LLMs:

### Ethical Considerations for LLMs

When working with LLMs, even smaller ones, it's important to consider several ethical aspects:

1. **Bias and Fairness**: LLMs can inherit biases from their training data, potentially reinforcing stereotypes or producing unfair outcomes.

2. **Privacy Concerns**: Models might memorize and reproduce sensitive information from their training data.

3. **Misinformation**: LLMs can generate plausible-sounding but incorrect information.

4. **Energy Consumption**: Training and running LLMs requires significant computational resources and energy.

5. **Appropriate Use Cases**: Consider whether an LLM is the appropriate tool for a given task.

### Mitigation Strategies:

- **Red-teaming and Adversarial Testing**: Proactively identify harmful outputs
- **Fine-tuning with Human Feedback**: Improve model behavior through feedback
- **Clear Documentation**: Document model limitations and intended uses
- **Content Filtering**: Implement filters for harmful content
- **User Education**: Educate users about model limitations

## 10. Conclusion and Best Practices

Let's summarize what we've learned and provide some best practices:

### Best Practices for Working with LLMs

1. **Start Small**: Use the smallest model that meets your requirements to minimize computational resources.

2. **Optimize for Efficiency**: Use techniques like quantization, caching, and batching to improve performance.

3. **Prompt Engineering**: Carefully design prompts to get the best results from the model.

4. **Evaluate Thoroughly**: Benchmark your model on relevant tasks before deployment.

5. **Monitor and Log**: Continuously monitor model outputs and performance in production.

6. **Establish Feedback Mechanisms**: Create ways for users to report problematic outputs.

7. **Set Appropriate Expectations**: Be transparent about the model's capabilities and limitations.

8. **Stay Updated**: The field is evolving rapidly, so stay informed about new techniques and models.

### Key Takeaways

- Smaller LLMs (1-7B parameters) can be powerful and efficient for many applications
- Understanding and tuning generation parameters is crucial for getting optimal results
- Different models excel at different tasks, so choose the right model for your use case
- Responsible use requires considering ethical implications and implementing safeguards

## 11. Further Resources

### Libraries and Tools:
- [Hugging Face Transformers](https://huggingface.co/docs/transformers/index)
- [bitsandbytes](https://github.com/TimDettmers/bitsandbytes) - Quantization library
- [PEFT](https://github.com/huggingface/peft) - Parameter-Efficient Fine-Tuning
- [LangChain](https://langchain.com/) - Building applications with LLMs

### Datasets:
- [OpenLLM Leaderboard](https://huggingface.co/spaces/HuggingFaceH4/open_llm_leaderboard)
- [MMLU Benchmark](https://github.com/hendrycks/test)
- [BIG-bench](https://github.com/google/BIG-bench)

### Articles and Papers:
- [Attention Is All You Need](https://arxiv.org/abs/1706.03762) - Original Transformer paper
- [Training language models to follow instructions](https://arxiv.org/abs/2203.02155) - InstructGPT paper
- [Scaling Laws for Neural Language Models](https://arxiv.org/abs/2001.08361) - OpenAI paper on scaling

In [ ]:
print("Thank you for completing this Hands-On Lab on Working with HuggingFace LLMs!")